# Temporal Harmonization and Alignment 

**Part 3 of the AQ Integration module.**

In Parts 1 and 2, **satellite** and **reanalysis** values were extracted at
the station locations.  Three kinds of data are now available, each with a
distinct time axis:

| Source | Typical cadence | Time-zone | Stamp semantics |
|---|---|---|---|
| Ground sensors | 1-minute / hourly | Often local (CET/CEST) | Period-start *or* period-end |
| CAMS European reanalysis | hourly | UTC | Period-start (analysis time) |
| ERA5 reanalysis | hourly | UTC | Instantaneous *or* hourly accumulation |

Before any model can consume them, they must share **one consistent time
axis**.  

Upon completion of this notebook, you will be able to:

1. Standardise a time column to **UTC**, including DST transitions.
2. Determine and document **timestamp semantics** (period-start versus period-end).
3. **Resample** minute-level observations to hourly values using explicit
   averaging conventions.
4. Construct a **master hourly index** per location and align every source to it.
5. Produce **integration-stage diagnostics**: coverage, missingness,
   duplicates, and gaps.

---

## The Four Time-Axis Pitfalls

### Time Zones and DST

Ground stations frequently log measurements in **local time**.  In Croatia,
local time corresponds to **CET (UTC+1) in winter and CEST (UTC+2) in
summer**, and the clock changes at the following transitions:

- **Last Sunday of March, 02:00 CET -> 03:00 CEST**  (spring forward, an
  hour that *never exists*).
- **Last Sunday of October, 03:00 CEST -> 02:00 CET**  (fall back, an hour
  that *exists twice*).

If you join two tz-naive series across a DST boundary, one of them is
silently shifted by an hour.  **Practical guideline.** Convert
**everything to UTC at the point of ingestion** and retain UTC until the
moment of plotting.

### Timestamp Semantics

A row stamped *10:00* can carry any of the following meanings:

- the **average over 10:00 – 10:59** (period-start label, the default in
  this notebook),
- the **average over 09:00 – 10:00** (period-end label, used by some
  meteorological vendors),
- the **instantaneous reading at exactly 10:00:00**.

Two systems using identical labels but different semantics are offset by
an hour after a naive join.  This notebook standardises on
**period-start, closed on the left**.

### Resampling Frequency

Models require a single cadence.  Minute-level sensors must therefore be
aggregated to hourly values.  The appropriate aggregation depends on the
underlying physics:

| Quantity | Aggregation |
|---|---|
| Concentration (PM, NO₂, O₃) | `mean` |
| Precipitation, traffic counts | `sum` |
| Wind gust, NO₂ peak | `max` |

Down-sampling is destructive; the choice of aggregation must be recorded
in the metadata.

### Merging Different Frequencies

Even when all sources nominally operate at the same cadence (hourly ERA5,
hourly CAMS reanalysis, and 1-minute stations resampled to hourly), they
must still share **the same hours**.  Sensor outages, late uploads, and
analysis-time shifts cause the timestamp sets to rarely match exactly.
The remedy is the **master hourly index**: a complete
`(time, location_id)` skeleton onto which every source is *reindexed*.
Downstream joins then reduce to a plain `pandas.merge`.

### Integration-Stage Diagnostics

Once alignment has been performed, and before any model training, the
following questions must be addressed:

- What **coverage** does each source achieve?
- Where are the **gaps** located?
- Are there **duplicate** timestamps?

These diagnostics are inexpensive to compute and costly to omit.

## Environment Setup

In [4]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [5]:
_scripts = Path.cwd().parent.parent / "scripts" / "part3_temporal"
if not _scripts.exists():
    _scripts = Path.cwd().parent / "scripts" / "part3_temporal"  # fallback
sys.path.insert(0, str(_scripts))

import time_align as ta
import temporal_diagnostics as td

## Generate a Synthetic Minute-Level Station Dataset

Real station files are rarely tidy.  To make the pitfalls concrete, this
notebook generates a one-month, 1-minute PM₂.₅ time series for the five
Croatian cities (June 2024, matching the CAMS and ERA5 download windows),
and then deliberately injects the kinds of problems that occur in
practice:

- timestamps **in local time** (Europe/Zagreb), not UTC,
- random **gaps** (sensor outages),
- a small number of **duplicate** timestamps (sensor restarts),
- a per-station offset combined with a realistic diurnal cycle.

The identical pipeline operates on any real CSV; in that case, omit this
section and load the real file instead.

In [9]:
rng = np.random.default_rng(42)

stations = pd.read_csv("../../data/stations_example.csv")
period_start_local = pd.Timestamp("2024-06-01 00:00:00", tz="Europe/Zagreb")
period_end_local   = pd.Timestamp("2024-06-30 23:59:00", tz="Europe/Zagreb")

minute_axis_local = pd.date_range(
    start=period_start_local, end=period_end_local, freq="1min"
)

# Diurnal cycle: low at night, peak in morning + evening commute hours.
def diurnal(t: pd.DatetimeIndex) -> np.ndarray:
    h = t.hour + t.minute / 60.0
    morning = np.exp(-((h - 8) / 2.0) ** 2)
    evening = np.exp(-((h - 19) / 2.0) ** 2)
    return 5.0 + 8.0 * morning + 10.0 * evening

base = diurnal(minute_axis_local)

records = []
for _, station in stations.iterrows():
    offset = rng.uniform(-1.5, 2.5)
    noise = rng.normal(0.0, 0.8, size=len(minute_axis_local))
    pm25 = np.clip(base + offset + noise, 0.5, None)

    records.append(pd.DataFrame({
        "time": minute_axis_local.tz_localize(None),
        "location_id": station["location_id"],
        "lat": station["lat"],
        "lon": station["lon"],
        "variable": "pm2p5",
        "value": pm25,
        "source": "ground_station_synth",
        "method": "raw_1min",
    }))

raw = pd.concat(records, ignore_index=True)

for loc in raw["location_id"].unique():
    mask_loc = raw["location_id"] == loc
    idx_local = raw.index[mask_loc]
    for _ in range(120):
        i0 = int(rng.integers(0, len(idx_local) - 60))
        length = int(rng.integers(10, 60))
        raw = raw.drop(idx_local[i0:i0 + length], errors="ignore")
raw = raw.reset_index(drop=True)

# Inject a handful of duplicate timestamps for one station.
dup = raw[raw["location_id"] == "ZAGREB01"].sample(10, random_state=0)
raw = pd.concat([raw, dup], ignore_index=True).sort_values(
    ["location_id", "time"]
).reset_index(drop=True)

print(f"Rows generated: {len(raw):,}")
print(f"Time-zone aware? {raw['time'].dt.tz is not None}  "
      f"({'tz-naive local time' if raw['time'].dt.tz is None else raw['time'].dt.tz})")
raw.head()


Rows generated: 196,074
Time-zone aware? False  (tz-naive local time)


,time,location_id,lat,lon,variable,value,source,method
0,2024-06-01 00:00:00,OSIJEK01,45.5511,18.6939,pm2p5,6.164900,ground_station_synth,raw_1min
1,2024-06-01 00:01:00,OSIJEK01,45.5511,18.6939,pm2p5,3.673601,ground_station_synth,raw_1min
2,2024-06-01 00:02:00,OSIJEK01,45.5511,18.6939,pm2p5,3.532935,ground_station_synth,raw_1min
3,2024-06-01 00:03:00,OSIJEK01,45.5511,18.6939,pm2p5,3.986217,ground_station_synth,raw_1min
4,2024-06-01 00:04:00,OSIJEK01,45.5511,18.6939,pm2p5,4.523688,ground_station_synth,raw_1min


## Standardise the Time Zone

The synthetic file is **tz-naive local** (Europe/Zagreb) and must be
converted to UTC at the point of ingestion.

`time_align.standardize_timezone` performs two operations:

1. **Localising** the naive timestamps so that pandas registers them as
   being in `Europe/Zagreb`.
2. **Converting** the result to UTC.

Because June lies entirely within summer time (CEST = UTC+2), all
timestamps shift downward by 2 hours.

In [10]:
std = ta.standardize_timezone(
    raw,
    time_col="time",
    source_tz="Europe/Zagreb",
    target_tz="UTC",
)
print(f"Before: {raw['time'].iloc[0]}        (tz-naive, local time)")
print(f"After:  {std['time'].iloc[0]}  (UTC)")
std.head()

Before: 2024-06-01 00:00:00        (tz-naive, local time)
After:  2024-05-31 22:00:00+00:00  (UTC)


,time,location_id,lat,lon,variable,value,source,method
0,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,pm2p5,6.164900,ground_station_synth,raw_1min
1,2024-05-31 22:01:00+00:00,OSIJEK01,45.5511,18.6939,pm2p5,3.673601,ground_station_synth,raw_1min
2,2024-05-31 22:02:00+00:00,OSIJEK01,45.5511,18.6939,pm2p5,3.532935,ground_station_synth,raw_1min
3,2024-05-31 22:03:00+00:00,OSIJEK01,45.5511,18.6939,pm2p5,3.986217,ground_station_synth,raw_1min
4,2024-05-31 22:04:00+00:00,OSIJEK01,45.5511,18.6939,pm2p5,4.523688,ground_station_synth,raw_1min


## DST Aside: Behaviour at the Autumn Fall-Back

The June data does not cross a DST boundary, so the conversion was clean.
To demonstrate the failure mode, consider a small dataset that spans the
autumn 2024 fall-back, which occurs at **03:00 CEST -> 02:00 CET on
Sunday 27 October 2024**.

Local time **02:30** occurs *twice* on that night.  

In [11]:
dst_demo = pd.DataFrame({
    "time": pd.to_datetime([
        "2024-10-27 02:00:00",   # pre-DST
        "2024-10-27 02:30:00",   # ambiguous!
        "2024-10-27 02:30:00",   # ambiguous (the second one)
        "2024-10-27 03:00:00",   # post-DST
    ]),
    "value": [10.1, 10.4, 10.6, 9.8],
})

try:
    ta.standardize_timezone(dst_demo, source_tz="Europe/Zagreb")
except Exception as e:
    print(f"ambiguous='raise'  ->  {type(e).__name__}: {e}")

infer = ta.standardize_timezone(
    dst_demo, source_tz="Europe/Zagreb", ambiguous="infer"
)
print()
print("With ambiguous='infer' the two 02:30 rows are kept apart:")
print(infer.to_string(index=False))

ambiguous='raise'  ->  ValueError: Cannot infer dst time from 2024-10-27 02:00:00, try using the 'ambiguous' argument

With ambiguous='infer' the two 02:30 rows are kept apart:
                     time  value
2024-10-27 00:00:00+00:00   10.1
2024-10-27 00:30:00+00:00   10.4
2024-10-27 01:30:00+00:00   10.6
2024-10-27 02:00:00+00:00    9.8


## Timestamp Semantics: Period-Start versus Period-End

This is a subtle but consequential distinction.  When pandas resamples to
hourly resolution, `label="left"` places the **start of the interval** on
the row, whereas `label="right"` places the **end** of the interval on
the row.

In [14]:
toy = pd.DataFrame({
    "time":  pd.date_range("2024-06-15 10:00", "2024-06-15 11:59",
                           freq="15min", tz="UTC"),
    "value": [10, 12, 14, 16, 18, 20, 22, 24],
})

left  = (toy.set_index("time")["value"]
            .resample("1h", label="left",  closed="left").mean())
right = (toy.set_index("time")["value"]
            .resample("1h", label="right", closed="right").mean())


In [15]:
toy

,time,value
0,2024-06-15 10:00:00+00:00,10
1,2024-06-15 10:15:00+00:00,12
2,2024-06-15 10:30:00+00:00,14
3,2024-06-15 10:45:00+00:00,16
4,2024-06-15 11:00:00+00:00,18
5,2024-06-15 11:15:00+00:00,20
6,2024-06-15 11:30:00+00:00,22
7,2024-06-15 11:45:00+00:00,24


In [16]:

side_by_side = pd.DataFrame({
    "label=left  (10:00 = avg 10:00-10:59)":  left,
    "label=right (11:00 = avg 10:01-11:00)":  right,
})
side_by_side

,label=left (10:00 = avg 10:00-10:59),label=right (11:00 = avg 10:01-11:00)
time,,
2024-06-15 10:00:00+00:00,13.0,10.0
2024-06-15 11:00:00+00:00,21.0,15.0
2024-06-15 12:00:00+00:00,NaN,22.0


## Resample Minute to Hourly

The five stations multiplied by 30 days multiplied by 1440 minutes yield
a series that is too dense for a model.  Aggregate the data to hourly
means using period-start labels, closed on the left.

In [17]:
clean = std.drop_duplicates(subset=["time", "location_id", "variable"])
print(f"Rows after de-duplication: {len(clean):,}  (was {len(std):,})")

hourly = ta.resample_to_hourly(
    clean,
    time_col="time",
    value_col="value",
    group_cols=["location_id", "source", "variable", "lat", "lon", "method"],
    agg="mean",
)
hourly["method"] = "mean_1h_left"
hourly.head(6)

Rows after de-duplication: 196,064  (was 196,074)


,time,location_id,source,variable,lat,lon,method,value
0,2024-05-31 22:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,4.384636
1,2024-05-31 23:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,4.403205
2,2024-06-01 00:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,4.442551
3,2024-06-01 01:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,4.361992
4,2024-06-01 02:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,4.969800
5,2024-06-01 03:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,6.113048


In [19]:
daily = ta.resample_to_hourly(
    hourly,
    time_col="time",
    value_col="value",
    group_cols=["location_id", "source", "variable", "lat", "lon"],
    agg="mean",
    freq="1D",
)
daily["method"] = "mean_1D"
daily

,time,location_id,source,variable,lat,lon,value,method
0,2024-05-31 00:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,4.393920,mean_1D
1,2024-06-01 00:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,7.079559,mean_1D
2,2024-06-02 00:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,7.083872,mean_1D
3,2024-06-03 00:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,7.089533,mean_1D
4,2024-06-04 00:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,7.014514,mean_1D
...,...,...,...,...,...,...,...,...
150,2024-06-26 00:00:00+00:00,ZAGREB01,ground_station_synth,pm2p5,45.8150,15.9819,9.254128,mean_1D
151,2024-06-27 00:00:00+00:00,ZAGREB01,ground_station_synth,pm2p5,45.8150,15.9819,9.257032,mean_1D
152,2024-06-28 00:00:00+00:00,ZAGREB01,ground_station_synth,pm2p5,45.8150,15.9819,9.193827,mean_1D
153,2024-06-29 00:00:00+00:00,ZAGREB01,ground_station_synth,pm2p5,45.8150,15.9819,9.189909,mean_1D


## Build the Master Hourly Index

The central scaffold of the integration stage is now constructed: a
complete `(time, location_id)` skeleton at hourly resolution, covering
the full period of interest, with one row for every expected hour at
every station.  

In [20]:
master = ta.build_master_index(
    start=pd.Timestamp("2024-06-01 00:00", tz="UTC"),
    end=pd.Timestamp("2024-06-30 23:00", tz="UTC"),
    locations=stations,
    freq="1h",
    tz="UTC",
)
print(
    f"Master index: {len(master)} rows  =  "
    f"{master['time'].nunique()} hours x {master['location_id'].nunique()} stations"
)
master.head()

Master index: 3600 rows  =  720 hours x 5 stations


,time,location_id,lat,lon,station_name,country
0,2024-06-01 00:00:00+00:00,OSIJEK01,45.5511,18.6939,Osijek,HR
1,2024-06-01 00:00:00+00:00,RIJEKA01,45.3271,14.4422,Rijeka,HR
2,2024-06-01 00:00:00+00:00,SPLIT01,43.5081,16.4402,Split,HR
3,2024-06-01 00:00:00+00:00,ZADAR01,44.1194,15.2314,Zadar,HR
4,2024-06-01 00:00:00+00:00,ZAGREB01,45.8150,15.9819,Zagreb,HR


## Align the Station Hourly Data to the Master Index

Where the station provides data, the master row
receives it; where the station provides no data (at the injected gaps),
the value is missing. 

In [21]:
station_aligned = ta.align_to_master_index(
    hourly,
    master=master,
    time_col="time",
    location_id_col="location_id",
    tolerance="30min",
    method="nearest",
)
print(
    f"Aligned station rows: {len(station_aligned)}  "
    f"(of {len(master)} possible)"
)
station_aligned.head()

Aligned station rows: 3590  (of 3600 possible)


,time,location_id,source,variable,lat,lon,method,value
0,2024-06-01 00:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,4.442551
1,2024-06-01 01:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,4.361992
2,2024-06-01 02:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,4.969800
3,2024-06-01 03:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,6.113048
4,2024-06-01 04:00:00+00:00,OSIJEK01,ground_station_synth,pm2p5,45.5511,18.6939,mean_1h_left,8.943091


## Align the CAMS Satellite Output

The Part 1 CSV (`data/outputs/satellite_pm25_nearest.csv`) currently
carries the **1970-epoch bug** in its `time` column: `satellite_extract_points`
passed the CAMS float32 `hours-since-base` values directly to
`pd.to_datetime()`, which interpreted them as bare nanoseconds and placed
every row at 1970-01-01.

This is itself an integration-stage diagnostic.  Caught at the point of
ingestion, the real timestamps can be recovered because the *int64*
representation of each timestamp equals the original `hours-since-base`
value verbatim (1 hour-as-int corresponds to 1 nanosecond-as-int after
the conversion).  Re-basing on the analysis start date (June 1 2024,
taken from the download script's `DATE_RANGE`) restores correct UTC
timestamps.

If `satellite_extract_points._to_long_dataframe` is later patched to
decode the CAMS time axis correctly, this block becomes a no-op.

In [22]:
SAT_CSV = Path("../../data/outputs/satellite_pm25_nearest.csv")

if SAT_CSV.exists():
    sat = pd.read_csv(SAT_CSV)
    sat["time"] = pd.to_datetime(sat["time"])

    if sat["time"].dt.year.iloc[0] == 1970:
        hours = sat["time"].astype("int64")
        sat["time"] = (
            pd.Timestamp("2024-06-01", tz="UTC")
            + pd.to_timedelta(hours, unit="h")
        )
        print(
            f"Fixed CAMS time axis: {sat['time'].min()} - {sat['time'].max()} "
            f"({sat['time'].nunique()} unique timestamps)"
        )
    else:
        sat["time"] = sat["time"].dt.tz_localize("UTC")

    sat_aligned = ta.align_to_master_index(
        sat, master=master, tolerance="30min",
    )
    print(f"Aligned CAMS rows: {len(sat_aligned)}")
    sat_aligned.head()
else:
    print(f"Part 1 file not found at {SAT_CSV} - skipping CAMS alignment.")
    sat_aligned = pd.DataFrame()

Aligned CAMS rows: 3600


## Align the  Reanalysis Output 

In [23]:
REA_CSV = Path("../../data/outputs/reanalysis_features.csv")

if REA_CSV.exists():
    rea = pd.read_csv(REA_CSV)
    rea["time"] = pd.to_datetime(rea["time"], utc=True)
    rea_aligned = ta.align_to_master_index(
        rea, master=master, tolerance="30min"
    )
    print(
        f"Aligned reanalysis rows: {len(rea_aligned)}  "
        f"(variables: {rea_aligned['variable'].nunique()})"
    )
    rea_aligned.head()
else:
    print(f"Part 2 file not found at {REA_CSV} - skipping reanalysis alignment.")
    rea_aligned = pd.DataFrame()

Aligned reanalysis rows: 25200  (variables: 7)


## Integration-Stage Diagnostics

Stack every aligned source into a single long table and execute the four
diagnostics provided by `temporal_diagnostics.py`:

- **coverage** — the fraction of expected master cells filled per
  source-location.
- **missingness by day** — the temporal distribution of the holes.
- **duplicates** — any `(time, location, variable)` triple appearing
  twice.
- **gaps** — the largest stretches of consecutive missing data.

In [24]:
all_aligned = pd.concat(
    [df for df in (station_aligned, sat_aligned, rea_aligned) if not df.empty],
    ignore_index=True,
)

report = td.summarize(
    all_aligned,
    master=master,
    expected_freq="1h",
)
td.print_summary(report)


=== COVERAGE ===
                   source location_id  observed  expected  coverage
CAMS European AQ analysis    OSIJEK01       720       720  1.000000
CAMS European AQ analysis    RIJEKA01       720       720  1.000000
CAMS European AQ analysis     SPLIT01       720       720  1.000000
CAMS European AQ analysis     ZADAR01       720       720  1.000000
CAMS European AQ analysis    ZAGREB01       720       720  1.000000
     ground_station_synth    OSIJEK01       717       720  0.995833
     ground_station_synth    RIJEKA01       718       720  0.997222
     ground_station_synth     SPLIT01       716       720  0.994444
     ground_station_synth     ZADAR01       716       720  0.994444
     ground_station_synth    ZAGREB01       716       720  0.994444
               reanalysis    OSIJEK01       720       720  1.000000
               reanalysis    RIJEKA01       720       720  1.000000
               reanalysis     SPLIT01       720       720  1.000000
               reanalysis     

## Save the Master Index and the Aligned Long-Format Table

The next module (spatial matching) joins on `(time, location_id)`, and
therefore requires exactly the artefacts now produced:

- `master_hourly_index.csv` — the empty skeleton.
- `aligned_long.csv` — every source and every variable on that skeleton.

In [25]:
out_dir = Path("../../data/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

master.to_csv(out_dir / "master_hourly_index.csv", index=False)
all_aligned.to_csv(out_dir / "aligned_long.csv", index=False)

print(f"Wrote: {out_dir / 'master_hourly_index.csv'}")
print(f"Wrote: {out_dir / 'aligned_long.csv'}")

Wrote: ..\..\data\outputs\master_hourly_index.csv
Wrote: ..\..\data\outputs\aligned_long.csv


In [26]:
# Quick wide-format preview: one column per variable, easy to scan.
wide = ta.to_wide(all_aligned)
print(f"Wide table shape: {wide.shape}")
print(f"Variables: {[c for c in wide.columns if c not in ('time','location_id','lat','lon')]}")
wide.head(8)

Wide table shape: (3600, 11)
Variables: ['blh', 'd2m', 'pm2p5', 'sp', 't2m', 'tp', 'wind_speed_10m']


,time,location_id,lat,lon,blh,d2m,pm2p5,sp,t2m,tp,wind_speed_10m
0,2024-06-01 00:00:00+00:00,OSIJEK01,45.5511,18.6939,53.547312,287.945847,4.442551,99849.159319,288.607564,0.000000,0.874448
1,2024-06-01 00:00:00+00:00,RIJEKA01,45.3271,14.4422,203.930549,283.793603,6.985451,95189.377304,284.916519,0.000448,1.821627
2,2024-06-01 00:00:00+00:00,SPLIT01,43.5081,16.4402,292.782244,290.233756,6.421498,98282.845689,292.513107,0.000271,3.662241
3,2024-06-01 00:00:00+00:00,ZADAR01,44.1194,15.2314,146.076319,288.017408,6.768171,99788.944013,290.975509,0.000227,2.066118
4,2024-06-01 00:00:00+00:00,ZAGREB01,45.8150,15.9819,259.557787,285.592837,6.592628,98562.004560,285.867271,0.000000,1.389644
5,2024-06-01 01:00:00+00:00,OSIJEK01,45.5511,18.6939,41.588441,287.400710,4.361992,99853.607286,287.777111,0.000000,1.134361
6,2024-06-01 01:00:00+00:00,RIJEKA01,45.3271,14.4422,292.062903,283.689003,7.009054,95206.840510,284.682293,0.000882,2.029562
7,2024-06-01 01:00:00+00:00,SPLIT01,43.5081,16.4402,270.673594,290.127087,6.356584,98310.675239,292.243565,0.000151,3.278663


## Summary

You have now constructed the temporal backbone of the integration stage:

1. Generated a realistic, irregular synthetic station dataset
   (minute-level, tz-naive local time, with gaps and duplicates).
2. Standardised the time zone to UTC, supported by a worked DST aside.
3. Examined and applied **period-start** timestamp semantics.
4. Resampled the data from minute to hourly resolution using explicit
   averaging conventions.
5. Constructed a master `(time, location_id)` hourly index.
6. Reindexed the Part 1 CAMS output (correcting the 1970-epoch bug) and
   the Part 2 reanalysis output onto this index.
7. Produced integration-stage diagnostics: coverage, missingness,
   duplicates, and gaps.
8. Saved the master index and the aligned long-format table.

**Next.** The spatial-matching module attaches static spatial features
(land cover, distance to roads, population density) to every
`(time, location_id)` row and prepares the final model-ready table.